In [1]:
pip install pandas sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --------------- ------------------------ 1.0/2.8 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 10.0 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
os.getcwd()

'C:\\Users\\HP\\OneDrive\\Desktop\\retail-bi-agent-final\\etl'

In [33]:
from sqlalchemy import create_engine, text

DB_USER = "postgres.qvarbiaokmhqataajpvj"
DB_PASSWORD = "dvijemacke123"
DB_HOST = "aws-1-eu-central-1.pooler.supabase.com"
DB_PORT = "5432"
DB_NAME = "postgres"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"sslmode": "require"}
)

In [23]:
df = pd.read_csv(CSV_PATH, encoding="ISO-8859-1")

In [25]:
df = df.rename(columns={
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country"
})

df = df.dropna(subset=["description", "customer_id"])
df = df[df["quantity"] > 0]
df = df[df["unit_price"] > 0]
df = df[~df["invoice_no"].astype(str).str.startswith("C")]

df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["customer_id"] = df["customer_id"].astype(int).astype(str)
df["total_amount"] = df["quantity"] * df["unit_price"]

print("Cleaned rows:", len(df))
df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_27304\847464594.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["invoice_date"] = pd.to_datetime(df["invoice_date"])


Cleaned rows: 397885


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,total_amount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [27]:
dim_date = df[["invoice_date"]].drop_duplicates().copy()
dim_date["full_date"] = dim_date["invoice_date"]
dim_date["year"] = dim_date["invoice_date"].dt.year
dim_date["month"] = dim_date["invoice_date"].dt.month
dim_date["day"] = dim_date["invoice_date"].dt.day
dim_date["hour"] = dim_date["invoice_date"].dt.hour
dim_date["day_of_week"] = dim_date["invoice_date"].dt.day_name()
dim_date = dim_date[["full_date", "year", "month", "day", "hour", "day_of_week"]]

dim_product = df[["stock_code", "description"]].drop_duplicates().copy()
dim_customer = df[["customer_id", "country"]].drop_duplicates().copy()

print(len(dim_date), len(dim_product), len(dim_customer))

17282 3897 4346


In [57]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT NOW();"))
    print(result.fetchone())

(datetime.datetime(2026, 6, 28, 21, 25, 35, 801707, tzinfo=datetime.timezone.utc),)


In [37]:
dim_date.to_sql("dim_date", engine, if_exists="append", index=False)
dim_product.to_sql("dim_product", engine, if_exists="append", index=False)
dim_customer.to_sql("dim_customer", engine, if_exists="append", index=False)

print("Dimensions loaded.")

Dimensions loaded.


In [39]:
dim_date = df[["invoice_date"]].drop_duplicates().copy()
dim_date["full_date"] = dim_date["invoice_date"]
dim_date["year"] = dim_date["invoice_date"].dt.year
dim_date["month"] = dim_date["invoice_date"].dt.month
dim_date["day"] = dim_date["invoice_date"].dt.day
dim_date["hour"] = dim_date["invoice_date"].dt.hour
dim_date["day_of_week"] = dim_date["invoice_date"].dt.day_name()
dim_date = dim_date[["full_date", "year", "month", "day", "hour", "day_of_week"]]

dim_product = df[["stock_code", "description"]].drop_duplicates().copy()
dim_customer = df[["customer_id", "country"]].drop_duplicates().copy()

print("Date rows:", len(dim_date))
print("Product rows:", len(dim_product))
print("Customer rows:", len(dim_customer))

Date rows: 17282
Product rows: 3897
Customer rows: 4346


In [41]:
dim_date.to_sql("dim_date", engine, if_exists="append", index=False)
dim_product.to_sql("dim_product", engine, if_exists="append", index=False)
dim_customer.to_sql("dim_customer", engine, if_exists="append", index=False)

print("Dimensions loaded.")

Dimensions loaded.


In [43]:
date_db = pd.read_sql("SELECT * FROM dim_date", engine)
product_db = pd.read_sql("SELECT * FROM dim_product", engine)
customer_db = pd.read_sql("SELECT * FROM dim_customer", engine)

fact = df.merge(date_db, left_on="invoice_date", right_on="full_date", how="left")
fact = fact.merge(product_db, on=["stock_code", "description"], how="left")
fact = fact.merge(customer_db, on=["customer_id", "country"], how="left")

fact_sales = fact[[
    "invoice_no",
    "date_key",
    "product_key",
    "customer_key",
    "quantity",
    "unit_price",
    "total_amount"
]]

print("Fact rows:", len(fact_sales))
fact_sales.head()

Fact rows: 397885


,invoice_no,date_key,product_key,customer_key,quantity,unit_price,total_amount
0,536365,1,1,1,6,2.55,15.30
1,536365,1,2,1,6,3.39,20.34
2,536365,1,3,1,8,2.75,22.00
3,536365,1,4,1,6,3.39,20.34
4,536365,1,5,1,6,3.39,20.34


In [45]:
fact_sales.to_sql("fact_sales", engine, if_exists="append", index=False)

print("Fact table loaded.")

Fact table loaded.


In [47]:
tables = ["dim_date", "dim_product", "dim_customer", "fact_sales"]

for table in tables:
    result = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table}", engine)
    print(table, result["row_count"][0])

dim_date 17282
dim_product 3897
dim_customer 4346
fact_sales 397885


In [49]:
query = """
SELECT 
    SUM(total_amount) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS total_orders,
    COUNT(DISTINCT customer_key) AS total_customers,
    SUM(total_amount) / COUNT(DISTINCT invoice_no) AS average_order_value
FROM fact_sales;
"""

pd.read_sql(query, engine)

,total_revenue,total_orders,total_customers,average_order_value
0,8911425.9,18532,4346,480.866927


In [51]:
tables = ["dim_date", "dim_product", "dim_customer", "fact_sales"]

for table in tables:
    result = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table}", engine)
    print(table, result["row_count"][0])

dim_date 17282
dim_product 3897
dim_customer 4346
fact_sales 397885


In [59]:
query = """
SELECT 
    d.year,
    d.month,
    ROUND(SUM(f.total_amount), 2) AS monthly_revenue
FROM fact_sales f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month;
"""

pd.read_sql(query, engine)

,year,month,monthly_revenue
0,2010,12,572713.89
1,2011,1,569445.04
2,2011,2,447137.35
3,2011,3,595500.76
4,2011,4,469200.36
5,2011,5,678594.56
6,2011,6,661213.69
7,2011,7,600091.01
8,2011,8,645343.90
9,2011,9,952838.38


In [61]:
query = """
SELECT 
    p.description,
    ROUND(SUM(f.total_amount), 2) AS revenue
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
GROUP BY p.description
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,description,revenue
0,"PAPER CRAFT , LITTLE BIRDIE",168469.60
1,REGENCY CAKESTAND 3 TIER,142592.95
2,WHITE HANGING HEART T-LIGHT HOLDER,100448.15
3,JUMBO BAG RED RETROSPOT,85220.78
4,MEDIUM CERAMIC TOP STORAGE JAR,81416.73
5,POSTAGE,77821.96
6,PARTY BUNTING,68844.33
7,ASSORTED COLOUR BIRD ORNAMENT,56580.34
8,Manual,53779.93
9,RABBIT NIGHT LIGHT,51346.20


In [63]:
query = """
SELECT 
    c.country,
    ROUND(SUM(f.total_amount), 2) AS revenue
FROM fact_sales f
JOIN dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,country,revenue
0,United Kingdom,7308391.55
1,Netherlands,285446.34
2,EIRE,265545.90
3,Germany,228867.14
4,France,209042.05
5,Australia,138521.31
6,Spain,61577.11
7,Switzerland,56443.95
8,Belgium,41196.34
9,Sweden,38378.33


In [65]:
query = """
SELECT 
    c.customer_id,
    c.country,
    ROUND(SUM(f.total_amount), 2) AS revenue
FROM fact_sales f
JOIN dim_customer c ON f.customer_key = c.customer_key
GROUP BY c.customer_id, c.country
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql(query, engine)

,customer_id,country,revenue
0,14646,Netherlands,280206.02
1,18102,United Kingdom,259657.30
2,17450,United Kingdom,194550.79
3,16446,United Kingdom,168472.50
4,14911,EIRE,143825.06
5,12415,Australia,124914.53
6,14156,EIRE,117379.63
7,17511,United Kingdom,91062.38
8,16029,United Kingdom,81024.84
9,12346,United Kingdom,77183.60


In [69]:
query = """
SELECT 
    ROUND(SUM(total_amount), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS total_orders,
    COUNT(DISTINCT customer_key) AS total_customers,
    ROUND(SUM(total_amount) / COUNT(DISTINCT invoice_no), 2) AS average_order_value
FROM fact_sales;
"""

pd.read_sql(query, engine)

,total_revenue,total_orders,total_customers,average_order_value
0,8911425.9,18532,4346,480.87
